# Single File Redundant Averaging

**by Josh Dillon**, last updated September 7, 2026

This notebook applies the smoothed calibration solutions from
[calibration_smoothing](https://github.com/HERA-Team/hera_notebook_templates/blob/master/notebooks/phase_II/calibration_smoothing.ipynb)
to a single raw file and coherently averages the calibrated visibilities within redundant baseline
groups. It is patterned on H6C's `file_postprocessing.ipynb`, stripped of the abs-cal, diff,
delay-filtered, incoherently averaged, and baseline-selected products (none of which Phase II uses),
and rebuilt around the per-file `SNAPDecoherence` sidecars from
[file_sky_calibration](https://github.com/HERA-Team/hera_notebook_templates/blob/master/notebooks/phase_II/file_sky_calibration.ipynb):
calibration and averaging go through `apply_cal.calibrate_and_red_avg`, which cleans the fitted
decoherence staircase out of the gains, calibrates and averages one redundant group at a time with
inverse-variance noise weights from the calibrated autocorrelations, divides inter-SNAP
cross-correlations by their measured coherence factors (intra-SNAP baselines and autocorrelations
are exempt), and returns *effective* numbers of samples that make the standard noise prediction
exact for every averaged product. Any antenna identity relabelings recorded in the calibration are
applied to the raw data first.

Before averaging, each cross-correlation is screened for X-engine packet failures — 32-channel
chunks carrying stale data from another baseline, the pathology H6C's `file_calibration` caught
through even/odd differences, which Phase II lacks — via gross non-redundancy with its group, and
those chunks are flagged for that baseline only. The notebook also reports the array's remaining
non-redundancy after smoothing, per antenna and per baseline, to inform whether individual baselines
should ever be excluded from the averages (none are, currently).

Configuration comes from a TOML file (e.g.
`hera_pipelines/pipelines/phase_II/idr1/v1/analysis/phase_II_analysis.toml`) pointed to by the
`TOML_FILE` environment variable; if none is given, the default settings in the cells below are
used. Environment variables otherwise carry only paths and wrapper-level toggles.

When `SAVE_RESULTS` is `TRUE`, a file with the following default name is written alongside the
input, with its name derived from `SUM_FILE` by replacing `.uvh5`:

* `*.smooth_calibrated.red_avg.uvh5` — smooth-calibrated, decoherence-corrected, redundantly
  averaged visibilities: one per redundant group, keyed by the group's first baseline, with the same
  baselines in every file of the night. **Its `nsamples` are effective numbers of samples, not
  counts.**

A fully-flagged file still produces this file (with every visibility flagged), so downstream, an
absent file always means a failed job.

Here's a set of links to skip to particular figures:

• [Figure 1: X-Engine Packet Failures Flagged on Individual Baselines](#Figure-1:-X-Engine-Packet-Failures-Flagged-on-Individual-Baselines)

• [Figure 2: Redundant Averaging of the Most Redundant Baseline Groups](#Figure-2:-Redundant-Averaging-of-the-Most-Redundant-Baseline-Groups)

• [Figure 3: Effective Number of Samples as a Function of Baseline](#Figure-3:-Effective-Number-of-Samples-as-a-Function-of-Baseline)

• [Figure 4: Redundant-Baseline chi^2 per Antenna After Smoothing](#Figure-4:-Redundant-Baseline-chi^2-per-Antenna-After-Smoothing)

• [Figure 5: Non-Redundancy of Individual Baselines](#Figure-5:-Non-Redundancy-of-Individual-Baselines)

• [Figure 6: Delay Spectra of Redundantly Averaged Visibilities Compared to Predicted Noise](#Figure-6:-Delay-Spectra-of-Redundantly-Averaged-Visibilities-Compared-to-Predicted-Noise)

In [ ]:
import time
tstart = time.time()
!hostname
!date

In [ ]:
import os
os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'
import h5py
import hdf5plugin  # REQUIRED to have the compression plugins available
import toml
import json
import warnings
import numpy as np
from scipy import constants
import matplotlib
import matplotlib.pyplot as plt
from pyuvdata import UVData
from hera_cal import io, utils, redcal, apply_cal, datacontainer, vis_clean, noise
from hera_filters import dspec
from hera_qm.metrics_io import read_a_priori_ant_flags
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
%matplotlib inline

## Parse inputs and outputs

To use this notebook interactively, you will have to provide a sum filename path if none exists as
an environment variable. All other parameters have reasonable default values.

In [ ]:
# parse wrapper-level environment variables: paths, plus the save toggle
SUM_FILE = os.environ.get("SUM_FILE", None)
# SUM_FILE = '/lustre/aoc/projects/hera/phase-II-analysis/idr1/2459935/zen.2459935.38700.sum.uvh5'
TOML_FILE = os.environ.get("TOML_FILE", None)
# TOML_FILE = '/lustre/aoc/projects/hera/phase-II-analysis/idr1/src/hera_pipelines/pipelines/phase_II/idr1/v1/analysis/phase_II_analysis.toml'
SAVE_RESULTS = os.environ.get("SAVE_RESULTS", "TRUE").upper() == "TRUE"
# which [WorkFlow] action is running this notebook
ACTION = os.environ.get("ACTION", "FILE_RED_AVG_NOTEBOOK")

# default suffixes for the files this notebook reads or writes
SUM_SUFFIX = 'sum.uvh5'
SMOOTH_CAL_SUFFIX = 'sum.smooth.calfits'
DECOHERENCE_SUFFIX = 'sum.snap_decoherence.h5'
RED_AVG_SUFFIX = 'sum.smooth_calibrated.red_avg.uvh5'

# frequency dividing the low and high bands (inside the a priori flagged FM gap), shared
# across the pipeline via [GLOBAL_OPTS]
BAND_SPLIT_FREQ = 100.0  # in MHz

# default settings, overridden by the TOML's [FILE_RED_AVG_OPTS] section (if given)
INCLUDE_CROSS_POLS = False  # also average en/ne cross-correlations (and cross-polarized autocorrelations)
PACKET_NCHANS = 32  # channels per X-engine-to-catcher packet, the granularity of stale-packet failures
PACKET_NONREDUNDANCY_CUT = 10.0  # flag a baseline's packet chunk whose non-redundancy with its group exceeds this multiple of the baseline's typical chunk

toml_options = (toml.load(TOML_FILE) if TOML_FILE is not None else {})

# Take the suffixes this notebook reads or writes from [DATA_PRODUCTS], scoped by that
# section's produced_by/consumed_by wiring, and require that wiring to agree exactly with
# the defaults above. That way the arrows drawn on the pipeline flowchart cannot drift from
# what this notebook actually touches: adding an edge there without using the file here (or
# vice versa) fails loudly, in the first cell, rather than silently.
declared_suffixes = {name for name in list(globals()) if name.endswith('_SUFFIX')}
wired_suffixes = set()
for product, spec in toml_options.get('DATA_PRODUCTS', {}).items():
    producers, consumers = (spec.get(key, []) for key in ['produced_by', 'consumed_by'])
    producers = ([producers] if isinstance(producers, str) else producers)
    consumers = ([consumers] if isinstance(consumers, str) else consumers)
    if 'suffix' in spec and (ACTION in producers or ACTION in consumers):
        wired_suffixes.add(f'{product.upper()}_SUFFIX')
        globals()[f'{product.upper()}_SUFFIX'] = spec['suffix']
if toml_options:
    assert wired_suffixes == declared_suffixes, (
        f'[DATA_PRODUCTS] wires {sorted(wired_suffixes)} to {ACTION}, '
        f'but this notebook declares {sorted(declared_suffixes)}.')

for toml_section in ['GLOBAL_OPTS', 'FILE_RED_AVG_OPTS']:
    if toml_section in toml_options:
        print(f'Loading overrides from [{toml_section}] in {TOML_FILE}.')
        for key, val in toml_options[toml_section].items():
            globals()[key.upper()] = val

SMOOTH_CAL_FILE = SUM_FILE.replace(SUM_SUFFIX, SMOOTH_CAL_SUFFIX)
DECOHERENCE_FILE = SUM_FILE.replace(SUM_SUFFIX, DECOHERENCE_SUFFIX)
RED_AVG_FILE = SUM_FILE.replace(SUM_SUFFIX, RED_AVG_SUFFIX)
# the day's final flags, written by calibration_smoothing (the last stage that touches flags)
aposteriori_yaml_file = os.path.join(os.path.dirname(SUM_FILE), SUM_FILE.split('.')[-4] + '_aposteriori_flags.yaml')

for setting in ['SUM_FILE', 'TOML_FILE', 'SAVE_RESULTS', 'ACTION', 'SUM_SUFFIX', 'SMOOTH_CAL_SUFFIX',
                'DECOHERENCE_SUFFIX', 'RED_AVG_SUFFIX', 'BAND_SPLIT_FREQ', 'INCLUDE_CROSS_POLS', 'PACKET_NCHANS',
                'PACKET_NONREDUNDANCY_CUT', 'SMOOTH_CAL_FILE', 'DECOHERENCE_FILE', 'RED_AVG_FILE',
                'aposteriori_yaml_file']:
    print(f'{setting} = {eval(setting)}')

## Load calibration, decoherence, and data

The smoothed gains carry each file's fitted decoherence staircase (see `calibration_smoothing`), so
they and the per-file `sky.calfits` mean the same thing to `apply_cal.calibrate_and_red_avg`. Any
identity repairs made by `file_sky_calibration` are recorded in the calibration's `RELABELS` extra
keyword and applied to the raw data here — the same bookkeeping fix, no data values change — so that
the data's antenna numbers agree with the gains, the antenna positions, and the sidecar's antenna →
SNAP mapping. Antennas whose SNAP has no decoherence measurement in this file cannot have their
inter-SNAP baselines corrected, so they are flagged.

In [ ]:
hc = io.HERACal(SMOOTH_CAL_FILE)
gains, cal_flags, _, _ = hc.read()
relabels = {int(labeled): int(true_ant)
            for labeled, true_ant in json.loads(hc.extra_keywords.get('RELABELS', '{}')).items()}
sd = io.SNAPDecoherence.read(DECOHERENCE_FILE)
ALL_FLAGGED = bool(np.all([cal_flags[ant] for ant in cal_flags]))

In [ ]:
pols = (['ee', 'nn', 'en', 'ne'] if INCLUDE_CROSS_POLS else ['ee', 'nn'])
hd = io.HERADataFastReader(SUM_FILE)
try:
    data, flags, nsamples = hd.read(pols=pols)
except KeyError:
    # there's a problem with one of the data/flags/nsamples fields
    ALL_FLAGGED = True
    bls_in_data = [bl for bl in hd.bls if bl[2] in pols]
    data = datacontainer.DataContainer({bl: np.zeros((len(hd.times), len(hd.freqs)), dtype=complex) for bl in bls_in_data})
    flags = datacontainer.DataContainer({bl: np.ones((len(hd.times), len(hd.freqs)), dtype=bool) for bl in bls_in_data})
    nsamples = datacontainer.DataContainer({bl: np.zeros((len(hd.times), len(hd.freqs))) for bl in bls_in_data})
for bl in flags:
    flags[bl] |= (nsamples[bl] == 0)
del nsamples

assert np.allclose(sd.times, hd.times, rtol=0, atol=1e-8), f'{DECOHERENCE_FILE} does not cover the times in {SUM_FILE}.'
assert all(gains[ant].shape == (len(hd.times), len(hd.freqs)) for ant in gains), f'{SMOOTH_CAL_FILE} does not match the shape of {SUM_FILE}.'

In [ ]:
if len(relabels) > 0:
    def fix_key(bl):
        return (relabels.get(bl[0], bl[0]), relabels.get(bl[1], bl[1]), bl[2])
    data = datacontainer.DataContainer({fix_key(bl): data[bl] for bl in data})
    flags = datacontainer.DataContainer({fix_key(bl): flags[bl] for bl in flags})
    print('Identity repairs applied: ' + '; '.join(f'visibilities labeled {labeled} reassigned to antenna {true_ant}'
                                                  for labeled, true_ant in sorted(relabels.items())) + '.')

In [ ]:
uncorrectable = sorted(ant for ant in gains if not np.all(cal_flags[ant])
                       and sd.ant_to_SNAP_dict.get(ant[0]) not in sd.decoherence)
for ant in uncorrectable:
    cal_flags[ant][:] = True
if len(uncorrectable) > 0:
    print(f'Flagging {len(uncorrectable)} antpol(s) without a decoherence measurement in this file: {uncorrectable}')
ALL_FLAGGED = ALL_FLAGGED or bool(np.all([cal_flags[ant] for ant in cal_flags]))

In [ ]:
# every redundant group the data could populate, so that every file of the night writes the same
# baselines; groups whose antennas are all flagged for the whole night (per the a posteriori yaml)
# are dropped
reds = redcal.get_reds(hd.data_antpos, pols=pols, include_autos=True)
day_ex_ants = set(read_a_priori_ant_flags(aposteriori_yaml_file))
reds = [red for red in reds if any(not any(ant in day_ex_ants for ant in utils.split_bl(bl)) for bl in red)]
print(f'Averaging {len(reds)} redundant groups, excluding those made up entirely of the {len(day_ex_ants)} antpols flagged all night.')

In [ ]:
# for calibrating individual members of a group, the way calibrate_and_red_avg does it internally
clean_gains = sd.correct_gains({ant: gains[ant] for ant in gains if ant[0] in sd.ant_to_SNAP_dict})
unmeasured = {SNAP: np.repeat(np.isnan(p), sd.block_freqs.shape[1], axis=1) for SNAP, p in sd.decoherence.items()}
dt = np.median(np.diff(hd.times)) * 24 * 3600
df = np.median(np.diff(hd.freqs))

def calibrated_members(red):
    '''Calibrated, decoherence-corrected members of a redundant group, with flagged cells set to nan.'''
    members = [bl for bl in red if bl in data
               and all(ant in clean_gains and not np.all(cal_flags[ant]) for ant in utils.split_bl(bl))]
    if len(members) == 0:
        return {}
    group = datacontainer.DataContainer({bl: data[bl].copy() for bl in members})
    group_flags = {}
    for bl in members:
        ant_i, ant_j = utils.split_bl(bl)
        group_flags[bl] = flags[bl] | cal_flags[ant_i] | cal_flags[ant_j]
        if sd.ant_to_SNAP_dict[bl[0]] != sd.ant_to_SNAP_dict[bl[1]]:  # inter-SNAP: unmeasured blocks cannot be corrected
            group_flags[bl] = group_flags[bl] | unmeasured[sd.ant_to_SNAP_dict[bl[0]]] | unmeasured[sd.ant_to_SNAP_dict[bl[1]]]
    apply_cal.calibrate_in_place(group, clean_gains)
    sd.correct_in_place(group, data_flags=datacontainer.DataContainer(group_flags))
    return {bl: np.where(group_flags[bl], np.nan, group[bl]) for bl in members}

## Screen for X-engine packet failures on individual baselines

The X-engines send the catcher one UDP packet per (baseline, time sample, even/odd parity,
32-channel chunk), and a missed packet leaves the catcher slot's previous occupant — generally a
different baseline's visibility from ~15 s earlier, often an autocorrelation — to be written out as
if it were current. H6C's `file_calibration` caught the gross cases through their non-noise-like
even/odd differences; Phase II has no diffs. Since the sum is even + odd, a single stale parity
halves an autocorrelation but multiplies a cross-correlation by hundreds, so these events are
grossly non-redundant over exactly one packet chunk (or three, when a whole X-engine block is
lost).

Each co-polarized cross-correlation is calibrated (with decoherence corrected, as
`calibrate_and_red_avg` does it) and compared to the per-channel median visibility of its redundant
group — robust to one bad member in groups of three or more — in units of its noise variance. The
median of that deviation over each packet chunk is then compared to its median over the baseline's
other chunks, which is insensitive both to the antenna-level non-redundancy that inflates every
chunk alike and to narrowband RFI, which cannot move a chunk's median. Chunks exceeding
`PACKET_NONREDUNDANCY_CUT` times the baseline's typical level are flagged, for that baseline only,
before averaging. Stale packets in autocorrelations are not screened here: they only halve the
autocorrelation over a chunk, which is left to the per-file autocorrelation classifiers and which
only mildly misweights that antenna's baselines.

In [ ]:
# calibrated autocorrelations set each cross-correlation's noise variance, as in calibrate_and_red_avg
with np.errstate(divide='ignore', invalid='ignore'):
    cal_autos = datacontainer.DataContainer({bl: np.abs(data[bl]) / np.abs(clean_gains[utils.split_bl(bl)[0]])**2
                                             for bl in data if bl[0] == bl[1] and utils.split_bl(bl)[0] in clean_gains
                                             and utils.split_pol(bl[2])[0] == utils.split_pol(bl[2])[1]})
assert len(hd.freqs) % PACKET_NCHANS == 0, f'{len(hd.freqs)} channels is not a whole number of {PACKET_NCHANS}-channel packets.'
nchunks = len(hd.freqs) // PACKET_NCHANS

packet_ratio, packet_bad = {}, {}
screen_start = time.time()
for red in reds:
    if red[0][0] == red[0][1] or utils.split_pol(red[0][2])[0] != utils.split_pol(red[0][2])[1]:
        continue  # co-polarized cross-correlations only
    members = calibrated_members(red)
    if len(members) < 2:
        continue
    stack = np.array(list(members.values()))
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')  # all-nan slices and nan comparisons are expected at flagged cells
        group_median = np.nanmedian(stack.real, axis=0) + 1j * np.nanmedian(stack.imag, axis=0)
        for bl, vis in members.items():
            deviation = np.abs(vis - group_median)**2 / noise.predict_noise_variance_from_autos(bl, cal_autos, dt=dt, df=df)
            chunk_median = np.nanmedian(deviation.reshape(len(hd.times), nchunks, PACKET_NCHANS), axis=2)
            ratio = chunk_median / np.nanmedian(chunk_median, axis=1, keepdims=True)
            ratio[np.sum(np.isfinite(chunk_median), axis=1) < 4] = np.nan  # too few chunks to set the baseline's typical level
            packet_ratio[bl] = ratio.astype(np.float32)
            packet_bad[bl] = ratio > PACKET_NONREDUNDANCY_CUT
            if np.any(packet_bad[bl]):
                flags[bl] = flags[bl] | np.repeat(packet_bad[bl], PACKET_NCHANS, axis=1)

n_bad = sum(np.sum(bad) for bad in packet_bad.values())
print(f'Screened {len(packet_bad)} baselines in {(time.time() - screen_start) / 60:.2f} minutes: flagged {n_bad} '
      f'(baseline, integration, packet) chunks on {sum(np.any(bad) for bad in packet_bad.values())} baselines.')
for pol in ['ee', 'nn']:
    counts = {}
    for bl, bad in packet_bad.items():
        if bl[2] == pol and np.any(bad):
            for antnum in bl[:2]:
                counts[antnum] = counts.get(antnum, 0) + np.sum(bad)
    if len(counts) > 0:
        print(f'    {pol}: antennas in the most flagged chunks: '
              + ', '.join(f'{antnum} ({counts[antnum]})' for antnum in sorted(counts, key=counts.get, reverse=True)[:10]))

In [ ]:
def plot_packet_failures(max_ants=40):
    if len(packet_ratio) == 0:
        print('No redundant groups with two or more baselines were screened. Nothing to plot.')
        return

    fig, axes = plt.subplots(2, 2, figsize=(14, 10), dpi=150, gridspec_kw={'height_ratios': [1, 1.5]})
    for ax, pol in zip(axes[0], ['ee', 'nn']):
        ratios = np.concatenate([r.ravel() for bl, r in packet_ratio.items() if bl[2] == pol] + [np.array([], dtype=np.float32)])
        ratios = ratios[np.isfinite(ratios) & (ratios > 0)]
        ax.hist(np.log10(ratios), bins=np.arange(-2, 7, .05))
        ax.axvline(np.log10(PACKET_NONREDUNDANCY_CUT), c='r', ls='--', label='PACKET_NONREDUNDANCY_CUT')
        ax.set_yscale('log')
        ax.set_xlabel('log$_{10}$(Packet Chunk Non-Redundancy Relative to Baseline Median)')
        ax.set_ylabel('Number of (Baseline, Integration, Chunk)s')
        ax.set_title(f'{pol}-polarized')
        ax.legend()

    block_edges = hd.freqs[::sd.block_freqs.shape[1]] / 1e6
    for ax, pol in zip(axes[1], ['ee', 'nn']):
        counts = {}
        for bl, bad in packet_bad.items():
            if bl[2] == pol and np.any(bad):
                for antnum in bl[:2]:
                    counts[antnum] = counts.get(antnum, 0) + np.sum(bad, axis=0)
        ax.set_xlabel('Frequency (MHz)')
        ax.set_title(f'{pol}-polarized: flagged baseline-integrations per packet chunk')
        if len(counts) == 0:
            ax.text(.5, .5, 'No packet failures flagged.', transform=ax.transAxes, ha='center', va='center')
            continue
        antnums = sorted(counts, key=lambda antnum: -np.sum(counts[antnum]))[:max_ants]
        im = ax.imshow([counts[antnum] for antnum in antnums], aspect='auto', interpolation='none', cmap='viridis',
                       extent=[hd.freqs[0] / 1e6, hd.freqs[-1] / 1e6, len(antnums) - .5, -.5])
        ax.set_yticks(range(len(antnums)))
        ax.set_yticklabels(antnums, fontsize=7)
        ax.set_ylabel('Antenna')
        for edge in block_edges:
            ax.axvline(edge, c='w', lw=.5, alpha=.5)
        plt.colorbar(im, ax=ax, label='Flagged Baseline-Integrations')
    plt.tight_layout()

# *Figure 1: X-Engine Packet Failures Flagged on Individual Baselines*

Top: the distribution of the packet screening statistic — the median over a packet chunk of a
baseline's noise-normalized deviation from its redundant group's median visibility, relative to the
median of the same statistic over that baseline's other chunks — for every (baseline, integration,
chunk), with the cut marked. Bottom: for the antennas involved in the most flagged chunks, the
number of flagged baseline-integrations per packet chunk. A failing baseline lights up both of its
antennas once; a failing antenna (as in the H6C cases, where the losses followed antenna number)
lights up one row across many baselines; a failing X-engine lights up a column. Thin lines mark
X-engine block boundaries.

In [ ]:
plot_packet_failures()

## Calibrate and redundantly average

`apply_cal.calibrate_and_red_avg` does the work, one redundant group at a time:

* the sidecar's fitted staircase is cleaned out of the gains (`SNAPDecoherence.correct_gains`),
  which is what autocorrelations and intra-SNAP baselines — exempt from decoherence — need;
* every cross-correlation gets an inverse-variance noise weight from the calibrated
  autocorrelations, $\sigma^2 = A_i A_j / (\Delta t \, \Delta \nu)$, inflated by
  $1 / ((1 - p_i)(1 - p_j))^2$ for inter-SNAP baselines, whose correction amplifies noise along with
  signal;
* inter-SNAP cross-correlations are divided by their measured coherence factors $(1 - p_i)(1 - p_j)$
  (`apply_cal.correct_SNAP_decoherence_in_place`), with blocks where either SNAP's decoherence is
  unmeasured flagged;
* the returned `nsamples` are *effective* numbers of samples,
  $\bar{A}_i \bar{A}_j \sum w / (\Delta t \, \Delta \nu)$, defined so that the standard noise
  prediction from the averaged autocorrelations is exact for every averaged product (they exceed the
  member count when quiet antennas dominate the average and fall short of it for corrected
  baselines);
* each member's DoF-normalized scatter about its group mean is accumulated into a redundant-baseline
  $\chi^2$ per antenna and per baseline. Since the gains were fit to the sky model rather than to
  redundancy, this $\chi^2$ measures the array's intrinsic non-redundancy plus calibration error in
  units of thermal noise, and is not expected to be ~1.

The packet chunks flagged above enter through `data_flags`, so they are excluded from the averages,
the effective numbers of samples, and the $\chi^2$.

In [ ]:
red_avg_data, red_avg_flags, red_avg_nsamples, red_avg_meta = apply_cal.calibrate_and_red_avg(
    data, gains, reds, ant_flags=cal_flags, data_flags=flags, snap_decoherence=sd, dt=dt, df=df)
ALL_FLAGGED = ALL_FLAGGED or bool(np.all([red_avg_flags[bl] for bl in red_avg_flags]))
for pol in ['Jee', 'Jnn']:
    if np.any(np.isfinite(red_avg_meta['total_chisq'].get(pol, np.nan))):
        print(f'Median unflagged redundant-baseline chi^2 / DoF for {pol}: {np.nanmedian(red_avg_meta["total_chisq"][pol]):.3f}')

In [ ]:
def plot_red_avg_vis(pols_to_plot=['ee', 'nn']):
    if ALL_FLAGGED:
        print('All integrations are flagged. Nothing to plot.')
        return

    fig, axes = plt.subplots(2, 2, figsize=(14, 6), dpi=150, sharex='col', sharey='row', gridspec_kw={'hspace': 0, 'wspace': 0})
    for i, pol in enumerate(pols_to_plot):
        # the cross-correlation group with the most effective samples
        candidates = [red for red in reds if red[0][2] == pol and red[0][0] != red[0][1] and red[0] in red_avg_data]
        if len(candidates) == 0:
            continue
        red = max(candidates, key=lambda red: np.median(red_avg_nsamples[red[0]]))
        tind = np.argmin(np.all(red_avg_flags[red[0]], axis=1))  # first integration not entirely flagged
        members = calibrated_members(red)
        for bl in members:
            axes[0, i].plot(hd.freqs / 1e6, np.angle(members[bl][tind]), alpha=.5, lw=.5)
            axes[1, i].semilogy(hd.freqs / 1e6, np.abs(members[bl][tind]), alpha=.5, lw=.5)

        to_plot = np.where(red_avg_flags[red[0]][tind], np.nan, red_avg_data[red[0]][tind])
        n_members = sum(not np.all(np.isnan(members[bl][tind])) for bl in members)
        med_nsamples = np.nanmedian(np.where(red_avg_flags[red[0]][tind], np.nan, red_avg_nsamples[red[0]][tind]))
        axes[0, i].plot(hd.freqs / 1e6, np.angle(to_plot), lw=1, c='k')
        axes[1, i].semilogy(hd.freqs / 1e6, np.abs(to_plot), lw=1, c='k',
                            label=f'Baseline Group {(int(red[0][0]), int(red[0][1]), red[0][2])}:\n'
                                  f'{n_members} baselines, {med_nsamples:.1f} effective samples')
        axes[1, i].set_xlabel('Frequency (MHz)')
        axes[1, i].legend(loc='upper right')
    axes[0, 0].set_ylabel('Visibility Phase (radians)')
    axes[1, 0].set_ylabel('Visibility Amplitude (Jy)')
    plt.tight_layout()

# *Figure 2: Redundant Averaging of the Most Redundant Baseline Groups*

The calibrated, decoherence-corrected members (thin colored lines) and inverse-variance weighted
average (black) of the cross-correlation group with the most effective samples in each polarization,
at the first integration that is not entirely flagged. Phases are shown in the top row, amplitudes
in the bottom; ee-polarized visibilities are in the left column and nn-polarized in the right.

In [ ]:
plot_red_avg_vis()

In [ ]:
if INCLUDE_CROSS_POLS:
    plot_red_avg_vis(['en', 'ne'])

In [ ]:
def plot_red_avg_nsamples():
    if ALL_FLAGGED:
        print('All integrations are flagged. Nothing to plot.')
        return

    fig, axes = plt.subplots(2, 1, figsize=(14, 7), dpi=150, sharex=True, gridspec_kw={'hspace': 0})
    med_nsamples = {red[0]: np.nanmedian(np.where(red_avg_flags[red[0]], np.nan, red_avg_nsamples[red[0]]))
                    for red in reds if red[0] in red_avg_data and not np.all(red_avg_flags[red[0]])}
    for ax, pol in zip(axes, ['ee', 'nn']):
        bls_here = [bl for bl in med_nsamples if bl[2] == pol]
        if len(bls_here) > 0:
            blvecs = np.array([hd.antpos[bl[1]] - hd.antpos[bl[0]] for bl in bls_here])
            sca = ax.scatter(blvecs[:, 0], blvecs[:, 1], s=0)
            sca = ax.scatter(blvecs[:, 0], blvecs[:, 1], s=(100 * (600 / np.diff(ax.get_xlim())[0])**2), ec='k', linewidths=.5,
                             c=[med_nsamples[bl] for bl in bls_here], cmap='turbo',
                             norm=matplotlib.colors.LogNorm(vmin=1, vmax=max(med_nsamples.values())))
            ax.axis('equal')
        ax.set_xlabel('EW Baseline Vector (m)')
        ax.set_ylabel('NS Baseline Vector (m)')
        ax.text(.98, .94, f'{pol}-polarized', transform=ax.transAxes, va='top', ha='right', bbox=dict(facecolor='w', alpha=0.5))

    plt.tight_layout()
    fig.colorbar(sca, ax=axes, pad=.02, label='Median Effective Number of Samples')

# *Figure 3: Effective Number of Samples as a Function of Baseline*

The median (over unflagged times and channels) effective number of samples in each redundantly
averaged cross-correlation, as a function of baseline vector. Note that the split of the HERA core
produces more highly-sampled intra-sector baselines that are interspersed with the
less-highly-sampled inter-sector baselines off the main grid. Effective samples exceed the member
count when the average leans on quieter-than-typical antennas and fall short of it where inter-SNAP
baselines were corrected for decoherence.

In [ ]:
plot_red_avg_nsamples()

## Examine non-redundancy after smoothing

In [ ]:
def chisq_array_plot(chisq_per_ant, statistic, vmax=None):
    '''Array plot of a per-antenna chi^2 statistic, averaged over its unflagged times and channels.
    vmax=None picks a robust scale from the data.'''
    if ALL_FLAGGED:
        print('All integrations are flagged. Nothing to plot.')
        return
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        avgs = {ant: np.nanmean(np.where(cal_flags[ant], np.nan, cspa)) for ant, cspa in chisq_per_ant.items()}
    if vmax is None:
        vmax = max(2, np.nanpercentile([m for m in avgs.values() if np.isfinite(m)], 90))

    def _chisq_subplot(antnums, size=250):
        fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=150)
        for ax, pol in zip(axes, ['Jee', 'Jnn']):
            # invisible scatter of every antenna position so flagged (chi^2-less) antennas
            # still fall inside the axes limits
            ax.scatter([hd.antpos[antnum][0] for antnum in antnums],
                       [hd.antpos[antnum][1] for antnum in antnums], s=size, facecolors='none', edgecolors='none')
            finite = [(antnum, pol) for antnum in antnums if np.isfinite(avgs.get((antnum, pol), np.nan))]
            for antnum in antnums:
                ax.text(hd.antpos[antnum][0], hd.antpos[antnum][1], antnum, va='center', ha='center', fontsize=8,
                        c=('w' if (antnum, pol) in finite else 'r'))
            ax.axis('equal')
            ax.set_xlabel('East-West Position (meters)')
            ax.set_ylabel('North-South Position (meters)')
            ax.set_title(f'{pol[1:]}-pol Mean {statistic} / Antenna')
            if len(finite) == 0:
                continue
            scatter = ax.scatter([hd.antpos[ant[0]][0] for ant in finite], [hd.antpos[ant[0]][1] for ant in finite],
                                 s=size, c=[avgs[ant] for ant in finite], lw=.25, edgecolors='none', zorder=-1,
                                 norm=matplotlib.colors.LogNorm(vmin=1, vmax=vmax))
            plt.colorbar(scatter, ax=ax, extend='both')
        plt.tight_layout()

    _chisq_subplot([antnum for antnum in hd.data_ants if antnum < 320])
    outriggers = [antnum for antnum in hd.data_ants if antnum >= 320
                  and any(np.isfinite(avgs.get((antnum, pol), np.nan)) for pol in ['Jee', 'Jnn'])]
    if len(outriggers) > 0:
        _chisq_subplot(outriggers, size=400)

# *Figure 4: Redundant-Baseline chi^2 per Antenna After Smoothing*

The mean over unflagged times and channels of each antenna's DoF-normalized weighted scatter of its
calibrated, decoherence-corrected visibilities about their redundant-group means. Unlike the
per-file version in `file_sky_calibration`, this uses the smoothed gains, so it also includes
whatever real per-file gain structure smoothing removed. It is not expected to be ~1 (see above);
what matters is relative structure across the array. Antennas numbered in red are flagged.

In [ ]:
chisq_array_plot(red_avg_meta['chisq_per_ant'], 'Redundant-Baseline $\\chi^2$')

In [ ]:
def plot_baseline_nonredundancy(n_worst=10):
    if ALL_FLAGGED:
        print('All integrations are flagged. Nothing to plot.')
        return
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        bl_chisq = {bl: np.nanmean(c) for bl, c in red_avg_meta['chisq_per_bl'].items() if np.any(np.isfinite(c))}

    # each baseline's chi^2 relative to the median of its redundant group (groups of one carry no information)
    ratios, group_sizes = {}, {}
    for red in reds:
        members = [bl for bl in red if bl in bl_chisq]
        if len(members) > 1:
            group_median = np.median([bl_chisq[bl] for bl in members])
            for bl in members:
                ratios[bl] = bl_chisq[bl] / group_median
                group_sizes[bl] = len(members)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=150)
    for pol, color in [('ee', 'C0'), ('nn', 'C1')]:
        bls_here = [bl for bl in ratios if bl[2] == pol]
        if len(bls_here) == 0:
            continue
        lengths = [np.linalg.norm(hd.antpos[bl[1]] - hd.antpos[bl[0]]) for bl in bls_here]
        axes[0].scatter(lengths, [bl_chisq[bl] for bl in bls_here], s=4, alpha=.3, color=color, label=f'{pol}-polarized')
        axes[1].hist(np.log10([ratios[bl] for bl in bls_here]), bins=np.arange(-1.5, 2.5, .05), alpha=.5, color=color, label=f'{pol}-polarized')
    axes[0].set_yscale('log')
    axes[0].set_xlabel('Baseline Length (m)')
    axes[0].set_ylabel('Mean Unflagged Redundant-Baseline $\\chi^2$ / DoF')
    axes[0].legend()
    axes[1].set_yscale('log')
    axes[1].set_xlabel('log$_{10}$($\\chi^2$ / DoF Relative to Redundant Group Median)')
    axes[1].set_ylabel('Number of Baselines')
    axes[1].legend()
    plt.tight_layout()

    print('Baselines most discrepant with their redundant groups (chi^2 / DoF relative to the group median):')
    for bl in sorted(ratios, key=ratios.get, reverse=True)[:n_worst]:
        print(f'    {(int(bl[0]), int(bl[1]), bl[2])}: {ratios[bl]:.1f}x the median of its group of {group_sizes[bl]} '
              f'(chi^2 / DoF = {bl_chisq[bl]:.2f})')

# *Figure 5: Non-Redundancy of Individual Baselines*

Left: every averaged baseline's mean unflagged redundant-baseline chi^2 / DoF against its length.
Right: the same chi^2 relative to the median of its own redundant group, which isolates baselines
that disagree with their group-mates beyond the antenna-to-antenna and sky-dependent non-redundancy
they all share. The most discrepant baselines are listed above the figure. Because a single
discrepant member also pulls the group mean toward itself, its chi^2 can exceed its group-mates' by
at most ~(N-1)^2 for an N-member group, so this diagnostic is only informative for groups of several
baselines. Packet failures were already flagged by the screen above, so what remains here is the
array's intrinsic non-redundancy. No baselines are excluded on this basis (yet).

In [ ]:
plot_baseline_nonredundancy()

In [ ]:
def plot_delay_spectra(min_dly=150.0, horizon=1.0, standoff=0.0, eigenval_cutoff=1e-12):
    '''Delay spectra of the redundantly averaged autocorrelations and 1, 2, 4, and 8-unit EW baselines
    after a diagnostic delay filter (min_dly in ns, horizon, standoff in ns), per band, compared to the
    noise level predicted from the averaged autocorrelations and effective nsamples.'''
    if ALL_FLAGGED:
        print('All integrations are flagged. Nothing to plot.')
        return

    # baselines to plot, by prototype vector: the averaged autocorrelations and the EW baselines
    bls_to_plot = {}
    for red in reds:
        if red[0][2] not in ['ee', 'nn'] or red[0] not in red_avg_data or np.all(red_avg_flags[red[0]]):
            continue
        vec = hd.antpos[red[0][1]] - hd.antpos[red[0][0]]
        units = int(np.round(np.abs(vec[0]) / 14.6))
        if np.abs(vec[1]) < 1 and units in [0, 1, 2, 4, 8]:
            bls_to_plot[units, red[0][2]] = red[0]
    if len(bls_to_plot) == 0:
        print('No unflagged autocorrelations or EW baselines to plot.')
        return

    # bands: the unflagged channel range on either side of the split
    flag_waterfall = np.all([red_avg_flags[bl] for bl in bls_to_plot.values()], axis=0)
    unflagged_chans = np.flatnonzero(~np.all(flag_waterfall[~np.all(flag_waterfall, axis=1)], axis=0))
    bands = [slice(chans[0], chans[-1] + 1) for chans in [unflagged_chans[hd.freqs[unflagged_chans] < BAND_SPLIT_FREQ * 1e6],
                                                          unflagged_chans[hd.freqs[unflagged_chans] >= BAND_SPLIT_FREQ * 1e6]]
             if len(chans) > 0]

    for band in bands:
        display(HTML(f'<h2>Redundantly Averaged Delay Spectra: {hd.freqs[band][0] / 1e6:.2f} — {hd.freqs[band][-1] / 1e6:.2f} MHz</h2>'))
        fig, axes = plt.subplots(5, 2, figsize=(14, 10), dpi=150, sharex=True, sharey='row', gridspec_kw={'hspace': 0, 'wspace': 0})
        nchan = band.stop - band.start
        window = dspec.gen_window('bh', nchan)
        delays = np.fft.fftshift(np.fft.fftfreq(nchan, df)) * 1e9
        for (units, pol), bl in bls_to_plot.items():
            ax = axes[{0: 0, 1: 1, 2: 2, 4: 3, 8: 4}[units], int(pol == 'nn')]
            unflagged_ints = ~np.all(red_avg_flags[bl][:, band], axis=1)
            if not np.any(unflagged_ints):
                continue
            f = red_avg_flags[bl][unflagged_ints][:, band]
            d = np.where(f, 0, red_avg_data[bl][unflagged_ints][:, band])
            # noise variance per cell from the averaged autocorrelations and effective nsamples
            avg_auto = np.abs(red_avg_data[(bl[0], bl[0], pol)])[unflagged_ints][:, band]
            with np.errstate(all='ignore'):
                sigma2 = np.where(f, 0, avg_auto**2 / (dt * df * red_avg_nsamples[bl][unflagged_ints][:, band]))
                wgts = np.where(f, 0, 1 / sigma2)
            wgts /= np.mean(wgts[wgts > 0])  # avoid dynamic range issues

            # delay filter (inpainting for autocorrelations, foreground removal for cross-correlations)
            bl_len = np.linalg.norm((hd.antpos[bl[1]] - hd.antpos[bl[0]])[:2]) / constants.c
            filter_centers, filter_half_widths = vis_clean.gen_filter_properties(ax='freq', min_dly=min_dly, horizon=horizon,
                                                                                 standoff=standoff, bl_len=bl_len)
            d_mdl, _, _ = dspec.fourier_filter(hd.freqs[band], d, wgts=wgts, filter_centers=filter_centers,
                                               filter_half_widths=filter_half_widths, mode='dpss_solve', ridge_alpha=0,
                                               fit_intercept=False, eigenval_cutoff=[eigenval_cutoff],
                                               suppression_factors=[eigenval_cutoff], max_contiguous_edge_flags=nchan)
            filtered = (np.where(f, d_mdl, d) if units == 0 else np.where(f, 0, d - d_mdl))

            # RMS delay spectrum over unflagged integrations vs. the same for the predicted noise
            dfft = np.fft.fftshift(np.fft.fft(filtered * window, axis=1), axes=1) * df
            ax.semilogy(delays, np.sqrt(np.mean(np.abs(dfft)**2, axis=0)),
                        label=('Inpainted Autocorrelation' if units == 0 else 'Delay-Filtered Data'))
            if units > 0:
                noise_rms = df * np.sqrt(np.mean(np.sum(window**2 * sigma2, axis=1)))
                ax.axhline(noise_rms, c='k', ls=':', label='Predicted Noise')
                ax.axvline(1e9 * filter_half_widths[0], c='k', ls='--', lw=1, alpha=.5, label='Delay Filter Edge')
                ax.axvline(-1e9 * filter_half_widths[0], c='k', ls='--', lw=1, alpha=.5)
                ax.set_ylim(bottom=noise_rms / 20)
            vec = hd.antpos[bl[1]] - hd.antpos[bl[0]]
            ax.text(.97, .9, f'{pol}-polarized\n{np.abs(vec[0]):.1f} m East\n{vec[1]:.1f} m North', transform=ax.transAxes,
                    va='top', ha='right', bbox=dict(facecolor='w', alpha=0.5))
            if pol == 'ee' and units in [0, 1]:
                ax.legend(loc='upper left')
            if pol == 'ee':
                ax.set_ylabel('$|\\widetilde{V}|$ (Jy Hz)')
            ax.set_xlim([-2250, 2250])
        for ax in axes[-1]:
            ax.set_xlabel('Delay (ns)')
        plt.tight_layout()
        plt.show()

# *Figure 6: Delay Spectra of Redundantly Averaged Visibilities Compared to Predicted Noise*

Delay spectra (Blackman-Harris tapered, RMS over unflagged integrations) of the redundantly averaged
autocorrelations and 1, 2, 4, and 8-unit East-West baselines, in the low and high bands separately,
after a diagnostic delay filter (150 ns minimum, horizon + 0 ns standoff) that is a figure-only step
— the written data are unfiltered. Autocorrelations are inpainted at flagged channels rather than
filtered. The dotted line is the noise level predicted from the averaged autocorrelations and the
effective numbers of samples, which the filtered data should reach at high delays if calibration,
flagging, and the decoherence correction have left nothing else behind. Frequency ranges may extend
slightly beyond the bands shown, since flagged channels at band edges were excluded.

In [ ]:
plot_delay_spectra()

## Save redundantly averaged visibilities

Every group is written, keyed by its first baseline (the same in every file of the night); groups
with no usable members in this file get fully-flagged placeholders so that every file has the same
shape. The `nsamples` written are the effective numbers of samples from `calibrate_and_red_avg`.

In [ ]:
add_to_history = ('Produced by file_redundant_averaging notebook with the following environment:\n' + '=' * 65 + '\n'
                  + os.popen('conda env export').read() + '=' * 65
                  + '\nnsamples are effective numbers of samples (see hera_cal.apply_cal.calibrate_and_red_avg), not counts.')

def _empty_hd(hd, antpairs, pols):
    x_orientation = hd.telescope.get_x_orientation_from_feeds()
    new_uvd = UVData.new(freq_array=hd.freq_array,
                         polarization_array=[utils.polstr2num(p, x_orientation=x_orientation) for p in pols],
                         times=hd.times,
                         telescope=hd.telescope,
                         antpairs=antpairs,
                         empty=True)
    return io.to_HERAData(new_uvd)

if SAVE_RESULTS:
    shape = (len(hd.times), len(hd.freqs))
    out_data = {red[0]: (red_avg_data[red[0]] if red[0] in red_avg_data else np.zeros(shape, dtype=complex)) for red in reds}
    out_flags = {red[0]: (red_avg_flags[red[0]] if red[0] in red_avg_flags else np.ones(shape, dtype=bool)) for red in reds}
    out_nsamples = {red[0]: (red_avg_nsamples[red[0]] if red[0] in red_avg_nsamples else np.zeros(shape)) for red in reds}

    hd_out = io.HERAData(SUM_FILE)
    antpairs = sorted(set(red[0][:2] for red in reds))
    try:
        hd_out.read(bls=antpairs, polarizations=pols)
    except KeyError:
        # there's a problem with one of the data/flags/nsamples fields
        hd_out = _empty_hd(hd_out, antpairs, pols)
    hd_out.empty_arrays()
    hd_out.update(data=out_data, flags=out_flags, nsamples=out_nsamples)
    hd_out.pol_convention = hc.pol_convention
    hd_out.vis_units = hc.gain_scale
    if len(relabels) > 0:
        hd_out.extra_keywords['RELABELS'] = hc.extra_keywords['RELABELS']
    hd_out.history += add_to_history
    print(f'Now writing redundantly averaged calibrated visibilities to {RED_AVG_FILE}')
    hd_out.write_uvh5(RED_AVG_FILE, clobber=True, fix_autos=True)

## Metadata

In [ ]:
for repo in ['hera_cal', 'hera_qm', 'hera_filters', 'hera_notebook_templates', 'pyuvdata']:
    exec(f'from {repo} import __version__')
    print(f'{repo}: {__version__}')

In [ ]:
print(f'Finished execution in {(time.time() - tstart) / 60:.2f} minutes.')